# Problema do Caixeiro-Viajante (TSP)

Cinco estratégias para o TSP sobre o mapa da Romênia — BFS, DFS, custo uniforme, guloso e força bruta — com matriz de distâncias construída por Dijkstra e comparação de custo e complexidade.

**Técnica:** Busca exaustiva, heurística gulosa e Dijkstra  
**Referência:** Russell & Norvig, *Inteligência Artificial* (AIMA)  
**Contexto:** disciplina de Inteligência Artificial — Análise e Desenvolvimento de Sistemas, FATEC Taubaté

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devcauas/agentes-ia/blob/main/notebooks/04-caixeiro-viajante.ipynb)


In [2]:
# ============================================================
# PROBLEMA DO CAIXEIRO-VIAJANTE (TSP) - VERSÃO CORRIGIDA
# ============================================================

import heapq
from collections import deque
from itertools import permutations
import math

# ------------------------------------------------------------
# GRAFO: Mapa da Romênia
# ------------------------------------------------------------
romania_map = {
    'Arad':      [('Zerind', 75),  ('Sibiu', 140), ('Timisoara', 118)],
    'Zerind':    [('Arad', 75),    ('Oradea', 71)],
    'Oradea':    [('Zerind', 71),  ('Sibiu', 151)],
    'Sibiu':     [('Arad', 140),   ('Oradea', 151), ('Fagaras', 99), ('Rimnicu', 80)],
    'Timisoara': [('Arad', 118),   ('Lugoj', 111)],
    'Lugoj':     [('Timisoara', 111), ('Mehadia', 70)],
    'Mehadia':   [('Lugoj', 70),   ('Dobreta', 75)],
    'Dobreta':   [('Mehadia', 75), ('Craiova', 120)],
    'Craiova':   [('Dobreta', 120),('Pitesti', 138), ('Rimnicu', 146)],
    'Rimnicu':   [('Sibiu', 80),   ('Craiova', 146), ('Pitesti', 97)],
    'Fagaras':   [('Sibiu', 99),   ('Bucharest', 211)],
    'Pitesti':   [('Rimnicu', 97), ('Craiova', 138), ('Bucharest', 101)],
    'Bucharest': [('Fagaras', 211),('Pitesti', 101), ('Giurgiu', 90), ('Urziceni', 85)],
    'Giurgiu':   [('Bucharest', 90)],
    'Urziceni':  [('Bucharest', 85), ('Hirsova', 98), ('Vaslui', 142)],
    'Hirsova':   [('Urziceni', 98), ('Eforie', 86)],
    'Eforie':    [('Hirsova', 86)],
    'Vaslui':    [('Urziceni', 142), ('Iasi', 92)],
    'Iasi':      [('Vaslui', 92),  ('Neamt', 87)],
    'Neamt':     [('Iasi', 87)]
}

TSP_CITIES = ['Arad', 'Zerind', 'Sibiu', 'Timisoara', 'Oradea']

# ============================================================
# CORREÇÃO 1: Dijkstra para calcular menor caminho entre
# quaisquer duas cidades — torna o grafo "completo" para o TSP
# ============================================================

def dijkstra_shortest_path(graph, source):
    """Retorna dicionário com menor distância de source para todas as cidades."""
    dist = {city: float('inf') for city in graph}
    dist[source] = 0
    heap = [(0, source)]

    while heap:
        cost, u = heapq.heappop(heap)
        if cost > dist[u]:
            continue
        for v, w in graph.get(u, []):
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                heapq.heappush(heap, (dist[v], v))
    return dist

def build_complete_distance_matrix(cities, graph):
    """
    Constrói matriz de distâncias reais (via caminho mais curto)
    entre todas as cidades do TSP — resolve o problema de grafo
    incompleto que causava 'Sem solução encontrada'.
    """
    matrix = {}
    for city in cities:
        shortest = dijkstra_shortest_path(graph, city)
        for other in cities:
            matrix[(city, other)] = shortest.get(other, float('inf'))
    return matrix

def get_dist(a, b, matrix):
    return matrix.get((a, b), float('inf'))

# ============================================================
# UTILITÁRIOS
# ============================================================

def print_separator(char="=", length=62):
    print(char * length)

def print_state_example():
    print_separator()
    print("REPRESENTAÇÃO DE ESTADOS: Roteamento vs TSP")
    print_separator()

    print("\n📍 ROTEAMENTO SIMPLES (Arad → Bucharest):")
    print("   Estado = apenas cidade atual")
    simples = ['Arad', 'Sibiu', 'Fagaras', 'Bucharest']
    for i, e in enumerate(simples):
        label = "🟢 INICIAL " if i == 0 else ("🏁 OBJETIVO" if i == len(simples)-1 else "          ")
        print(f"   {label} → '{e}'")

    print("\n📍 TSP (Visitar todas as cidades e retornar):")
    print("   Estado = (cidade_atual, {cidades_visitadas})")
    tsp_states = [
        ('Arad',      frozenset(['Arad'])),
        ('Zerind',    frozenset(['Arad', 'Zerind'])),
        ('Oradea',    frozenset(['Arad', 'Zerind', 'Oradea'])),
        ('Sibiu',     frozenset(['Arad', 'Zerind', 'Oradea', 'Sibiu'])),
        ('Timisoara', frozenset(['Arad', 'Zerind', 'Oradea', 'Sibiu', 'Timisoara'])),
        ('Arad',      frozenset(['Arad', 'Zerind', 'Oradea', 'Sibiu', 'Timisoara'])),
    ]
    for i, (cidade, vis) in enumerate(tsp_states):
        label = "🟢 INICIAL  " if i == 0 else ("🏁 OBJETIVO " if i == len(tsp_states)-1 else f"   Passo {i}  ")
        print(f"   {label} → ({cidade}, {set(vis)})")

    print("\n💡 Por que incluir cidades visitadas?")
    print("   A mesma cidade pode ser alcançada por caminhos diferentes.")
    print("   Ex: Sibiu via Arad→Sibiu  difere de  Arad→Oradea→Sibiu no TSP!")

# ============================================================
# BUSCA EM LARGURA — BFS
# ============================================================

def tsp_bfs(cities, start, dist_matrix):
    all_cities = frozenset(cities)
    queue = deque([((start, frozenset([start])), [start], 0)])
    visited_states = set()
    nodes_explored = 0

    while queue:
        (cur, visited), path, cost = queue.popleft()
        nodes_explored += 1
        key = (cur, visited)
        if key in visited_states:
            continue
        visited_states.add(key)

        if visited == all_cities:
            rc = get_dist(cur, start, dist_matrix)
            if rc != float('inf'):
                return {'algorithm': 'BFS', 'path': path + [start],
                        'cost': cost + rc, 'nodes_explored': nodes_explored, 'optimal': False}

        for nxt in cities:
            if nxt not in visited:
                d = get_dist(cur, nxt, dist_matrix)
                if d != float('inf'):
                    new_state = (nxt, visited | frozenset([nxt]))
                    if new_state not in visited_states:
                        queue.append((new_state, path + [nxt], cost + d))
    return None

# ============================================================
# BUSCA EM PROFUNDIDADE — DFS
# ============================================================

def tsp_dfs(cities, start, dist_matrix):
    all_cities = frozenset(cities)
    stack = [((start, frozenset([start])), [start], 0)]
    visited_states = set()
    nodes_explored = 0

    while stack:
        (cur, visited), path, cost = stack.pop()
        nodes_explored += 1
        key = (cur, visited)
        if key in visited_states:
            continue
        visited_states.add(key)

        if visited == all_cities:
            rc = get_dist(cur, start, dist_matrix)
            if rc != float('inf'):
                return {'algorithm': 'DFS', 'path': path + [start],
                        'cost': cost + rc, 'nodes_explored': nodes_explored, 'optimal': False}

        neighbors = []
        for nxt in cities:
            if nxt not in visited:
                d = get_dist(cur, nxt, dist_matrix)
                if d != float('inf'):
                    new_state = (nxt, visited | frozenset([nxt]))
                    if new_state not in visited_states:
                        neighbors.append(((new_state), path + [nxt], cost + d))
        for item in reversed(neighbors):
            stack.append(item)
    return None

# ============================================================
# BUSCA DE CUSTO UNIFORME — UCS  ✅ garante ótimo
# ============================================================

def tsp_ucs(cities, start, dist_matrix):
    all_cities = frozenset(cities)
    counter = 0
    heap = [(0, counter, (start, frozenset([start])), [start])]
    best_cost = {}
    nodes_explored = 0

    while heap:
        cost, _, (cur, visited), path = heapq.heappop(heap)
        nodes_explored += 1
        key = (cur, visited)
        if key in best_cost and best_cost[key] <= cost:
            continue
        best_cost[key] = cost

        if visited == all_cities:
            rc = get_dist(cur, start, dist_matrix)
            if rc != float('inf'):
                return {'algorithm': 'UCS', 'path': path + [start],
                        'cost': cost + rc, 'nodes_explored': nodes_explored, 'optimal': True}

        for nxt in cities:
            if nxt not in visited:
                d = get_dist(cur, nxt, dist_matrix)
                if d != float('inf'):
                    new_cost = cost + d
                    new_key = (nxt, visited | frozenset([nxt]))
                    if new_key not in best_cost or best_cost[new_key] > new_cost:
                        counter += 1
                        heapq.heappush(heap, (new_cost, counter,
                                              (nxt, visited | frozenset([nxt])),
                                              path + [nxt]))
    return None

# ============================================================
# HEURÍSTICA GULOSA — NEAREST NEIGHBOR
# ============================================================

def tsp_greedy(cities, start, dist_matrix):
    current = start
    unvisited = set(cities) - {start}
    path = [start]
    total_cost = 0
    nodes_explored = 0

    while unvisited:
        nodes_explored += 1
        best_city = min(unvisited, key=lambda c: get_dist(current, c, dist_matrix))
        best_dist = get_dist(current, best_city, dist_matrix)
        if best_dist == float('inf'):
            return None
        path.append(best_city)
        total_cost += best_dist
        unvisited.remove(best_city)
        current = best_city

    rc = get_dist(current, start, dist_matrix)
    if rc == float('inf'):
        return None
    return {'algorithm': 'Greedy (Nearest Neighbor)', 'path': path + [start],
            'cost': total_cost + rc, 'nodes_explored': nodes_explored, 'optimal': False}

# ============================================================
# FORÇA BRUTA  ✅ garante ótimo
# ============================================================

def tsp_brute_force(cities, start, dist_matrix):
    others = [c for c in cities if c != start]
    best_cost = float('inf')
    best_path = None          # <-- CORREÇÃO 2: inicializa explicitamente
    total_routes = 0

    for perm in permutations(others):
        route = [start] + list(perm) + [start]
        total_routes += 1
        cost = 0
        valid = True
        for i in range(len(route) - 1):
            d = get_dist(route[i], route[i+1], dist_matrix)
            if d == float('inf'):
                valid = False
                break
            cost += d
        if valid and cost < best_cost:
            best_cost = cost
            best_path = route

    # CORREÇÃO 2: retorna None se nenhuma rota válida foi encontrada
    if best_path is None:
        return None

    return {'algorithm': 'Força Bruta', 'path': best_path,
            'cost': best_cost, 'nodes_explored': total_routes, 'optimal': True}

# ============================================================
# COMPLEXIDADE
# ============================================================

def analyze_complexity():
    print_separator()
    print("ANÁLISE DE COMPLEXIDADE DO TSP")
    print_separator()
    print(f"\n{'N cidades':<12} {'Rotas (N-1)!/2':<22} {'Estados N×2^N':<22}")
    print("-" * 58)
    for n in [3, 4, 5, 6, 8, 10, 12, 15, 20]:
        rotas   = math.factorial(n - 1) // 2
        estados = n * (2 ** n)
        r = f"{rotas:,}" if rotas < 1e15 else f"{rotas:.2e}"
        e = f"{estados:,}" if estados < 1e12 else f"{estados:.2e}"
        print(f"{n:<12} {r:<22} {e:<22}")
    print("\n📌 Rotas:  (N-1)! / 2   →  cresce fatorialmente (NP-difícil)")
    print("📌 Estados: N × 2^N     →  explosão combinatória")

# ============================================================
# COMPARAÇÃO FINAL
# ============================================================

def compare_algorithms(results):
    print_separator()
    print("COMPARAÇÃO DOS ALGORITMOS")
    print_separator()
    print(f"\n{'Algoritmo':<28} {'Custo':>8}  {'Nós':>8}  {'Ótimo?'}")
    print("-" * 58)
    for r in sorted(results, key=lambda x: x['cost']):
        otimo = "✅ Sim" if r['optimal'] else "❌ Não"
        print(f"{r['algorithm']:<28} {r['cost']:>8}  {r['nodes_explored']:>8}  {otimo}")
    best = min((r for r in results if r['optimal']), key=lambda x: x['cost'], default=None)
    if best:
        print(f"\n🏆 Rota ótima: {' → '.join(best['path'])}")
        print(f"   Distância : {best['cost']} km (via caminhos mais curtos do grafo)")

# ============================================================
# APLICAÇÕES REAIS
# ============================================================

def show_real_applications():
    print_separator()
    print("APLICAÇÕES REAIS DO TSP")
    print_separator()
    apps = [
        ("🔬 Sequenciamento de DNA",
         "Fragmentos de DNA → cidades; sobreposição → distância.",
         "Montar genoma inteiro minimizando erros de remontagem."),
        ("🖨️  Perfuração de PCB (placas eletrônicas)",
         "Furos da placa → cidades; tempo de deslocamento da broca → custo.",
         "Reduzir trajeto da broca economiza tempo e desgaste."),
        ("📦 Logística de entregas (Correios / Amazon)",
         "Endereços → cidades; distância entre pontos → custo.",
         "Rota otimizada reduz combustível e emissões de CO₂."),
        ("🔭 Posicionamento de telescópio",
         "Alvos celestes → cidades; tempo de reposicionamento → custo.",
         "Maximizar observações em uma única noite de trabalho."),
    ]
    for titulo, como, impacto in apps:
        print(f"\n{titulo}")
        print(f"   Como vira TSP : {como}")
        print(f"   Impacto       : {impacto}")

# ============================================================
# MAIN
# ============================================================

def main():
    print_separator()
    print("  PROBLEMA DO CAIXEIRO-VIAJANTE (TSP) — versão corrigida")
    print("  Mapa da Romênia · Busca Cega + Heurística")
    print_separator()

    print_state_example()
    analyze_complexity()

    # ── CORREÇÃO 1: matriz de distâncias reais (Dijkstra) ──────────
    print_separator()
    print(f"Construindo matriz de distâncias reais para: {TSP_CITIES}")
    dist_matrix = build_complete_distance_matrix(TSP_CITIES, romania_map)
    print("Matriz de distâncias (via caminho mais curto):")
    header = f"{'':>12}" + "".join(f"{c:>12}" for c in TSP_CITIES)
    print(header)
    for c1 in TSP_CITIES:
        row = f"{c1:>12}" + "".join(f"{dist_matrix[(c1,c2)]:>12.0f}" for c2 in TSP_CITIES)
        print(row)
    print_separator()

    start_city = TSP_CITIES[0]
    results = []

    algorithms = [
        ("BFS",         tsp_bfs),
        ("DFS",         tsp_dfs),
        ("UCS",         tsp_ucs),
        ("Greedy",      tsp_greedy),
        ("Força Bruta", tsp_brute_force),
    ]

    for name, func in algorithms:
        print(f"\n🔍 Executando {name}...")
        result = func(TSP_CITIES, start_city, dist_matrix)
        # ── CORREÇÃO 2: verifica None antes de acessar 'path' ──────
        if result is not None and result.get('path') is not None:
            results.append(result)
            print(f"   Rota          : {' → '.join(result['path'])}")
            print(f"   Custo total   : {result['cost']} km")
            print(f"   Nós explorados: {result['nodes_explored']}")
            print(f"   Garante ótimo : {'✅' if result['optimal'] else '❌'}")
        else:
            print(f"   ⚠️  Nenhuma rota válida encontrada.")

    if results:
        compare_algorithms(results)

    print_separator()
    print("ANÁLISE QUALITATIVA (Questão 4)")
    print_separator()
    print("""
🔵 BFS  → NÃO garante ótimo em grafos ponderados.
          Minimiza número de paradas, ignora pesos das arestas.

🟠 DFS  → NÃO garante ótimo. Depende da ordem de expansão;
          pode encontrar solução muito subótima ou demorar muito.

🟢 UCS  → GARANTE ótimo! Expande sempre o menor custo acumulado.
          Mais lento que BFS/DFS, mas correto para grafos ponderados.

⚡ Greedy → Muito rápido, sem garantia de ótimo. Na prática gera
           rotas ~20% acima do ótimo. Útil para N grande.

🔴 Força Bruta → GARANTE ótimo, mas O((N-1)!). Inviável para N>15.
""")

    show_real_applications()
    print_separator()
    print("FIM")
    print_separator()

if __name__ == "__main__":
    main()

  PROBLEMA DO CAIXEIRO-VIAJANTE (TSP) — versão corrigida
  Mapa da Romênia · Busca Cega + Heurística
REPRESENTAÇÃO DE ESTADOS: Roteamento vs TSP

📍 ROTEAMENTO SIMPLES (Arad → Bucharest):
   Estado = apenas cidade atual
   🟢 INICIAL  → 'Arad'
              → 'Sibiu'
              → 'Fagaras'
   🏁 OBJETIVO → 'Bucharest'

📍 TSP (Visitar todas as cidades e retornar):
   Estado = (cidade_atual, {cidades_visitadas})
   🟢 INICIAL   → (Arad, {'Arad'})
      Passo 1   → (Zerind, {'Zerind', 'Arad'})
      Passo 2   → (Oradea, {'Zerind', 'Arad', 'Oradea'})
      Passo 3   → (Sibiu, {'Zerind', 'Sibiu', 'Arad', 'Oradea'})
      Passo 4   → (Timisoara, {'Sibiu', 'Timisoara', 'Zerind', 'Arad', 'Oradea'})
   🏁 OBJETIVO  → (Arad, {'Sibiu', 'Timisoara', 'Zerind', 'Arad', 'Oradea'})

💡 Por que incluir cidades visitadas?
   A mesma cidade pode ser alcançada por caminhos diferentes.
   Ex: Sibiu via Arad→Sibiu  difere de  Arad→Oradea→Sibiu no TSP!
ANÁLISE DE COMPLEXIDADE DO TSP

N cidades    Rotas (N-1)!/2